# Python Logging Lab — ML Pipeline with Iris Dataset

This lab demonstrates Python's `logging` module applied to a real ML workflow.
Unlike a generic logging demo, here we log every step of a scikit-learn training
pipeline: data loading, preprocessing, model training, evaluation, and prediction.

**Modifications from base lab:**
- Applied logging to a real ML pipeline (not a toy example)
- Uses the Iris classification dataset
- Logs training metrics (accuracy, classification report) at INFO level
- Logs warnings for edge cases (e.g., unknown class predictions)
- Writes all logs to both console and a `ml_pipeline.log` file
- Uses a custom named logger (`ml_pipeline`) instead of root logger


In [1]:
import logging
import sys
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


### 1. Logger Setup — Dual Handler (Console + File)

We create a named logger `ml_pipeline` with two handlers:
- `StreamHandler` → prints to console/notebook output
- `FileHandler` → writes to `ml_pipeline.log`

This is more production-ready than using `logging.basicConfig()` directly.


In [2]:
# Create a named logger (not the root logger)
logger = logging.getLogger("ml_pipeline")
logger.setLevel(logging.DEBUG)

# Avoid duplicate handlers if the cell is re-run
if not logger.handlers:
    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)

    # File handler — writes all logs to ml_pipeline.log
    file_handler = logging.FileHandler("ml_pipeline.log")
    file_handler.setLevel(logging.DEBUG)

    # Shared formatter
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    console_handler.setFormatter(formatter)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

logger.info("Logger initialized. Dual output: console + ml_pipeline.log")


2026-03-14 11:11:06,452 - ml_pipeline - INFO - Logger initialized. Dual output: console + ml_pipeline.log


### 2. Load and Inspect Data


In [3]:
logger.info("Loading Iris dataset...")

try:
    iris = load_iris()
    X = pd.DataFrame(iris.data, columns=iris.feature_names)
    y = pd.Series(iris.target, name="species")
    logger.info(f"Dataset loaded successfully. Shape: {X.shape}")
    logger.debug(f"Feature names: {list(X.columns)}")
    logger.debug(f"Class distribution:\n{y.value_counts().to_dict()}")
    print(X.head())
except Exception as e:
    logger.exception(f"Failed to load dataset: {e}")
    raise


2026-03-14 11:11:06,456 - ml_pipeline - INFO - Loading Iris dataset...
2026-03-14 11:11:06,462 - ml_pipeline - INFO - Dataset loaded successfully. Shape: (150, 4)
2026-03-14 11:11:06,462 - ml_pipeline - DEBUG - Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
2026-03-14 11:11:06,464 - ml_pipeline - DEBUG - Class distribution:
{0: 50, 1: 50, 2: 50}
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2


### 3. Train/Test Split


In [4]:
logger.info("Splitting data into train and test sets (80/20)...")

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    logger.info(f"Train size: {X_train.shape[0]} samples | Test size: {X_test.shape[0]} samples")

    # Log a warning if test set is very small
    if X_test.shape[0] < 20:
        logger.warning(f"Test set is small ({X_test.shape[0]} samples). Results may not be reliable.")
    else:
        logger.debug(f"Test set size is adequate: {X_test.shape[0]} samples")

except Exception as e:
    logger.exception(f"Error during train/test split: {e}")
    raise


2026-03-14 11:11:06,470 - ml_pipeline - INFO - Splitting data into train and test sets (80/20)...
2026-03-14 11:11:06,473 - ml_pipeline - INFO - Train size: 120 samples | Test size: 30 samples
2026-03-14 11:11:06,474 - ml_pipeline - DEBUG - Test set size is adequate: 30 samples


### 4. Train RandomForest Model

**Modification from base lab:** The original lab has no model. Here we train a
`RandomForestClassifier` and log hyperparameters at DEBUG level.


In [5]:
n_estimators = 100
max_depth = 4

logger.info(f"Training RandomForestClassifier (n_estimators={n_estimators}, max_depth={max_depth})...")
logger.debug(f"Hyperparameters: n_estimators={n_estimators}, max_depth={max_depth}, random_state=42")

try:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    logger.info("Model training complete.")
except Exception as e:
    logger.critical(f"Model training failed: {e}")
    raise


2026-03-14 11:11:06,478 - ml_pipeline - INFO - Training RandomForestClassifier (n_estimators=100, max_depth=4)...
2026-03-14 11:11:06,479 - ml_pipeline - DEBUG - Hyperparameters: n_estimators=100, max_depth=4, random_state=42
2026-03-14 11:11:06,587 - ml_pipeline - INFO - Model training complete.


### 5. Evaluate the Model


In [6]:
logger.info("Evaluating model on test set...")

try:
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=iris.target_names)

    logger.info(f"Test Accuracy: {acc:.4f}")
    logger.debug(f"Classification Report:\n{report}")

    # Log a warning if accuracy drops below threshold
    if acc < 0.90:
        logger.warning(f"Accuracy {acc:.4f} is below the 90% threshold. Consider tuning the model.")
    else:
        logger.info("Accuracy is above the 90% threshold. Model performance is acceptable.")

    print(f"\nAccuracy: {acc:.4f}")
    print(f"\nClassification Report:\n{report}")

except Exception as e:
    logger.exception(f"Evaluation failed: {e}")
    raise


2026-03-14 11:11:06,590 - ml_pipeline - INFO - Evaluating model on test set...
2026-03-14 11:11:06,600 - ml_pipeline - INFO - Test Accuracy: 0.9667
2026-03-14 11:11:06,601 - ml_pipeline - DEBUG - Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

2026-03-14 11:11:06,601 - ml_pipeline - INFO - Accuracy is above the 90% threshold. Model performance is acceptable.

Accuracy: 0.9667

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy           

### 6. Log Feature Importances (DEBUG level)


In [7]:
importances = dict(zip(iris.feature_names, model.feature_importances_))
logger.debug("Feature importances:")
for feat, imp in sorted(importances.items(), key=lambda x: -x[1]):
    logger.debug(f"  {feat}: {imp:.4f}")

print("Feature Importances:")
for feat, imp in sorted(importances.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.4f}")


2026-03-14 11:11:06,612 - ml_pipeline - DEBUG - Feature importances:
2026-03-14 11:11:06,613 - ml_pipeline - DEBUG -   petal length (cm): 0.4377
2026-03-14 11:11:06,614 - ml_pipeline - DEBUG -   petal width (cm): 0.4359
2026-03-14 11:11:06,617 - ml_pipeline - DEBUG -   sepal length (cm): 0.1156
2026-03-14 11:11:06,619 - ml_pipeline - DEBUG -   sepal width (cm): 0.0108
Feature Importances:
  petal length (cm): 0.4377
  petal width (cm): 0.4359
  sepal length (cm): 0.1156
  sepal width (cm): 0.0108


### 7. Predict on New Sample + Log Edge Case Handling


In [8]:
# Sample a new unseen data point — wrap in DataFrame with proper column names
new_sample = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=iris.feature_names  # matches training feature names → fixes UserWarning
)

logger.info(f"Running prediction on new sample: {new_sample.values.tolist()}")

try:
    prediction = model.predict(new_sample)[0]
    proba = model.predict_proba(new_sample)[0]
    predicted_class = iris.target_names[prediction]

    logger.info(f"Predicted class: {predicted_class} (index {int(prediction)})")
    
    # Convert numpy types → plain Python floats for clean log output
    clean_proba = {str(iris.target_names[i]): round(float(p), 4) for i, p in enumerate(proba)}
    logger.debug(f"Class probabilities: {clean_proba}")

    max_confidence = float(max(proba))
    if max_confidence < 0.70:
        logger.warning(f"Low confidence prediction: {max_confidence:.2f}. Treat result with caution.")
    else:
        logger.info(f"High confidence prediction: {max_confidence:.2f}")

    print(f"\nPredicted species: {predicted_class}")
    print(f"Confidence: {max_confidence:.2%}")

except Exception as e:
    logger.exception(f"Prediction failed: {e}")
    raise


2026-03-14 11:11:06,633 - ml_pipeline - INFO - Running prediction on new sample: [[5.1, 3.5, 1.4, 0.2]]
2026-03-14 11:11:06,648 - ml_pipeline - INFO - Predicted class: setosa (index 0)
2026-03-14 11:11:06,649 - ml_pipeline - DEBUG - Class probabilities: {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}
2026-03-14 11:11:06,650 - ml_pipeline - INFO - High confidence prediction: 1.00

Predicted species: setosa
Confidence: 100.00%


### 8. Verify Log File Was Written


In [9]:
logger.info("Pipeline complete. Reading log file contents...")

print("\n--- Contents of ml_pipeline.log ---\n")
with open("ml_pipeline.log", "r") as f:
    print(f.read())


2026-03-14 11:11:06,652 - ml_pipeline - INFO - Pipeline complete. Reading log file contents...

--- Contents of ml_pipeline.log ---

2026-03-14 10:59:01,247 - ml_pipeline - INFO - Logger initialized. Dual output: console + ml_pipeline.log
2026-03-14 10:59:39,525 - ml_pipeline - INFO - Loading Iris dataset...
2026-03-14 10:59:39,650 - ml_pipeline - INFO - Dataset loaded successfully. Shape: (150, 4)
2026-03-14 10:59:39,652 - ml_pipeline - DEBUG - Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
2026-03-14 10:59:39,671 - ml_pipeline - DEBUG - Class distribution:
{0: 50, 1: 50, 2: 50}
2026-03-14 11:00:06,095 - ml_pipeline - INFO - Splitting data into train and test sets (80/20)...
2026-03-14 11:00:06,476 - ml_pipeline - INFO - Train size: 120 samples | Test size: 30 samples
2026-03-14 11:00:06,476 - ml_pipeline - DEBUG - Test set size is adequate: 30 samples
2026-03-14 11:00:35,786 - ml_pipeline - INFO - Training RandomForestClassifier (n_e

## Summary

| Concept | Where Used in This Lab |
|---|---|
| Named logger | `logging.getLogger("ml_pipeline")` |
| DEBUG level | Hyperparameters, class distribution, feature importances, probabilities |
| INFO level | Dataset shape, split sizes, accuracy, training complete |
| WARNING level | Small test set, low accuracy, low confidence |
| CRITICAL level | Model training crash handler |
| Exception logging | `logger.exception(...)` in all try/except blocks |
| StreamHandler | Console/notebook output |
| FileHandler | `ml_pipeline.log` persistent log file |

**Key difference from original lab:** This lab applies logging to a real ML pipeline on the Iris dataset,
demonstrating how logging is actually used in production ML code.
